In [7]:
import re
import pandas as pd
df = pd.read_excel('/Users/juliakulpa/Desktop/milliken_28_04/asia_28_04/BOM_asia_28_04.xlsx')
col_CAS = "Tier {i} Material is CAS?"

def get_highest_tier(df, col_pattern):
    numbers = []

    # Convert pattern into regex
    regex_pattern = col_pattern.replace("{i}", r"(\d+)")
    print(regex_pattern)
    regex_pattern = f"{regex_pattern}"
    print(regex_pattern)

    for col in df.columns:
        match = re.match(regex_pattern, col)
        if match:
            numbers.append(int(match.group(1)))

    if numbers:
        return max(numbers)
    else:
        print("Not determined max tier, set it to 10")
        return 10

get_highest_tier(df, col_CAS)

Tier (\d+) Material is CAS?
Tier (\d+) Material is CAS?


5

In [20]:
import pandas as pd
import numpy as np
import itertools

df = pd.read_excel('/Users/juliakulpa/Library/CloudStorage/Dropbox-Arche/Julia Kulpa/milliken/Milliken_BOM.xlsx')
df.head(10)
print(df.dtypes)

Product                                                                          str
Min weight Homogenous material in Product                                    float64
Max weight Homogenous material in Product                                    float64
Min % Homogenous material in Product                                         float64
Max % Homogenous material in Product                                         float64
                                                                              ...   
Coupled to Tier 5 material (only present if coupled material is present)     float64
Going to another TRL?.1                                                      float64
CAS Tier 5                                                                       str
Documets.2                                                                   float64
Notes.2                                                                          str
Length: 74, dtype: object


In [21]:
col_min_perc = "Tier {i} Material Weight% Min"
col_max_perc = "Tier {i} Material Weight% Max"

tier_level = 5

for i in range(2, tier_level + 1):
    min_col = col_min_perc.format(i=i)
    max_col = col_max_perc.format(i=i)
    print(df[[min_col,min_col]].nunique())

Tier 2 Material Weight% Min    78
Tier 2 Material Weight% Min    78
dtype: int64
Tier 3 Material Weight% Min    320
Tier 3 Material Weight% Min    320
dtype: int64
Tier 4 Material Weight% Min    96
Tier 4 Material Weight% Min    96
dtype: int64
Tier 5 Material Weight% Min    26
Tier 5 Material Weight% Min    26
dtype: int64


In [22]:
def clean_numeric_series(series, col_name=None):
    # Step 1: normalize basic formatting
    s = series.astype(str).str.strip().str.replace(",", ".", regex=False)

    # Step 2: detect non-numeric BEFORE coercion
    numeric_check = pd.to_numeric(s, errors="coerce")
    mask_bad = numeric_check.isna() & s.notna() & (s != "")

    if mask_bad.any():
        print(f"Non-numeric values found in column: {col_name}")
        print(s[mask_bad].unique())

    # Step 3: clean problematic characters (light cleaning only)
    s_clean = (
        s.str.replace("%", "", regex=False)
         .str.replace("<", "", regex=False)
         .str.replace(">", "", regex=False)
    )

    # Step 4: convert to numeric
    result = pd.to_numeric(s_clean, errors="coerce")

    # Step 5: enforce float64
    return result.astype("float64")
    # columns that must be numeric
numeric_cols = [
    col_min_perc,
    col_max_perc,
]

for i in range(1, tier_level + 1):
    min_col = col_min_perc.format(i=i)
    if min_col in df.columns:
        df[min_col] = clean_numeric_series(df[min_col])
    max_col = col_max_perc.format(i=i)
    if max_col in df.columns:
        df[max_col] = clean_numeric_series(df[max_col])



⚠️ Non-numeric values found in column: None
<StringArray>
['?']
Length: 1, dtype: str

⚠️ Non-numeric values found in column: None
<StringArray>
['?']
Length: 1, dtype: str

⚠️ Non-numeric values found in column: None
<StringArray>
['?']
Length: 1, dtype: str

⚠️ Non-numeric values found in column: None
<StringArray>
['?']
Length: 1, dtype: str


In [23]:
for i in range(2, tier_level + 1):
    min_col = col_min_perc.format(i=i)
    max_col = col_max_perc.format(i=i)
    print(df[[min_col,min_col]].nunique())


Tier 2 Material Weight% Min    77
Tier 2 Material Weight% Min    77
dtype: int64
Tier 3 Material Weight% Min    319
Tier 3 Material Weight% Min    319
dtype: int64
Tier 4 Material Weight% Min    95
Tier 4 Material Weight% Min    95
dtype: int64
Tier 5 Material Weight% Min    26
Tier 5 Material Weight% Min    26
dtype: int64


In [ ]:
    def a():# Normalize
        colours = df[colour_col].astype(str).str.upper().str.strip()
        conc = df[conc_col].fillna(0)

        # Sums needed for the decision tree
        red_ge_1 = conc[(colours == "RED") & (conc >= 1)].sum()
        red_ge_01_lt_1 = conc[(colours == "RED") & (conc >= 0.1) & (conc < 1)].sum()
        grey_ge_01 = conc[(colours == "GREY") & (conc >= 0.1)].sum()
        yellow_ge_1 = conc[(colours == "YELLOW") & (conc >= 1)].sum()

        # 1. RED present at >= 1%, sum >= 5%
        if red_ge_1 >= 5:
            return "RED"

        # 2. RED >=1% and <5% + GREY >=0.1%, sum >=5%
        if red_ge_1 + grey_ge_01 >= 5:
            return "GREY"

        # 3. RED present at >=1%, sum >=1%
        if red_ge_1 >= 1:
            return "YELLOW"

        # 4. 10 x RED >=0.1% and <1% + YELLOW >=1%, sum >=1%
        if (10 * red_ge_01_lt_1) + yellow_ge_1 >= 1:
            return "YELLOW"

        return "GREEN"